# 04 — Find & Add Visit-Level Clinical Diagnosis to Master Table

This notebook identifies the clinical diagnosis (`CN`, `MCI`, `AD`) for all **3,278 fMRI–T1w scan sessions** in `results/dataset_creation/clean_dedup_fmri_to_t1_master.csv`, adds a `Diagnosis` column to the master CSV, and performs a rigorous cross-source comparison across all candidate tables and columns in the workspace:

1. **Primary Source Selected — `data/DXSUM_15Oct2026.csv` (`DIAGNOSIS` column):**
   - Official ADNI **Diagnostic Summary (`DXSUM`)** table covering all study phases (`ADNI1`, `ADNIGO`, `ADNI2`, `ADNI3`, `ADNI4`, `TEAM`) from `2005-09-29` through `2026-10-14`.
   - `DIAGNOSIS` codes: `1.0 = CN` (Cognitively Normal), `2.0 = MCI` (Mild Cognitive Impairment), `3.0 = AD` (Dementia / Alzheimer's Disease), `10.0 = TEAM_NODX`.
2. **Alternative Sources Evaluated & Why They Were Not Used as Primary:**
   - `clean_dedup_fmri_to_t1_master.csv` (`Group` column from LONI IDA search export): Static baseline enrollment cohort (`0 / 1,329` subjects ever change `Group` across longitudinal visits); misses **452 longitudinal conversions/reversions (13.80%)**.
   - `data/other data - maybe could be used/ADNIMERGE_15Oct2026.csv` (`DX` and `DX_bl` columns): Stops on `2023-09-06` (contains **0 `ADNI4` rows**) and has `4,963` `NaN` values in `DX`, leaving **42.13% (`1,381 / 3,278`)** of our scans without an exact-visit diagnosis. However, where `ADNIMERGE.DX` is present (`1,897` rows), it has **100.00% (`1,897 / 1,897`) agreement** with `DXSUM.DIAGNOSIS`.
   - `DXSUM_15Oct2026.csv` legacy columns (`DXNORM`, `DXMCI`, `DXAD`): Only populated in `ADNI1` (`0%` coverage in `ADNIGO/ADNI2/ADNI3/ADNI4`).
   - `data/other data - maybe could be used/CDR_15Oct2026.csv` (`CDGLOBAL` column): Functional staging score rather than consensus clinical diagnosis (`87.89%` agreement with `DXSUM`, but classifies 135 mild AD dementia cases with `CDR=0.5` as MCI and 125 MCI cases with `CDR=0.0` as CN). Used only as a fallback for 2 `ADNIGO` screening-withdrawal subjects (`031_S_2017`, `130_S_2402`) and 2 `TEAM-ADNI` baseline subjects (`010_S_12018`, `070_S_12043`).
   - `BLCHANGE_15Oct2026.csv`, `PREPARE_Phase1_data`, and `tadpole_challenge/ADNIMERGE.csv`: Either lack diagnostic class labels (`BLCHANGE`), cover only baseline (`PREPARE`), or are frozen in 2017 (`TADPOLE`, missing 72.8% of scans).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"d:\Mahdi\OU\Alzheimer's Disease\gnn-connectivity-matrix")
MASTER_CSV    = PROJECT_ROOT / "results" / "dataset_creation" / "clean_dedup_fmri_to_t1_master.csv"
DXSUM_CSV     = PROJECT_ROOT / "data" / "DXSUM_15Oct2026.csv"
DATADIC_CSV   = PROJECT_ROOT / "data" / "DATADIC_19Oct2026.csv"
OTHER_DIR     = PROJECT_ROOT / "data" / "other data - maybe could be used"
ADNIMERGE_CSV = OTHER_DIR / "ADNIMERGE_15Oct2026.csv"
CDR_CSV       = OTHER_DIR / "CDR_15Oct2026.csv"
MMSE_CSV      = OTHER_DIR / "MMSE_15Oct2026.csv"
BLCHANGE_CSV  = OTHER_DIR / "BLCHANGE_15Oct2026.csv"
PREPARE_CSV   = OTHER_DIR / "PREPARE_Phase1_data" / "ADNI_target_and_other_predictor_N1526_20240620.csv"
TADPOLE_CSV   = OTHER_DIR / "tadpole_challenge" / "ADNIMERGE.csv"

master_df   = pd.read_csv(MASTER_CSV)
dxsum_df    = pd.read_csv(DXSUM_CSV, low_memory=False)
adnimerge_df = pd.read_csv(ADNIMERGE_CSV, low_memory=False)
cdr_df      = pd.read_csv(CDR_CSV, low_memory=False)
mmse_df     = pd.read_csv(MMSE_CSV, low_memory=False)
blchange_df = pd.read_csv(BLCHANGE_CSV, low_memory=False)
prepare_df  = pd.read_csv(PREPARE_CSV, low_memory=False)
tadpole_df  = pd.read_csv(TADPOLE_CSV, low_memory=False)

print(f"Master table shape:    {master_df.shape} | StudyDate range: {master_df['StudyDate'].min()} -> {master_df['StudyDate'].max()}")
print(f"DXSUM table shape:     {dxsum_df.shape} | EXAMDATE range:  {dxsum_df['EXAMDATE'].dropna().min()} -> {dxsum_df['EXAMDATE'].dropna().max()}")
print(f"ADNIMERGE table shape: {adnimerge_df.shape} | EXAMDATE range:  {adnimerge_df['EXAMDATE'].dropna().min()} -> {adnimerge_df['EXAMDATE'].dropna().max()}")
print(f"CDR table shape:       {cdr_df.shape} | VISDATE range:   {cdr_df['VISDATE'].dropna().min()} -> {cdr_df['VISDATE'].dropna().max()}")
print(f"TADPOLE table shape:   {tadpole_df.shape} | EXAMDATE range:  {tadpole_df['EXAMDATE'].dropna().min()} -> {tadpole_df['EXAMDATE'].dropna().max()}")
print(f"PREPARE table shape:   {prepare_df.shape}")

Master table shape:    (3278, 32) | StudyDate range: 2010-06-24 -> 2026-09-28
DXSUM table shape:     (16546, 41) | EXAMDATE range:  2005-09-29 -> 2026-10-14
ADNIMERGE table shape: (16421, 116) | EXAMDATE range:  2005-09-07 -> 2023-09-06
CDR table shape:       (15099, 25) | VISDATE range:   2005-08-17 -> 2026-10-13
TADPOLE table shape:   (13915, 113) | EXAMDATE range:  2005-09-07 -> 2019-04-09
PREPARE table shape:   (1526, 10)


## 1. Survey of All Candidate Diagnosis Columns Across CSV Files

First, we inspect the coverage of candidate diagnosis columns inside `DXSUM_15Oct2026.csv`, `ADNIMERGE_15Oct2026.csv`, `clean_dedup_fmri_to_t1_master.csv`, `CDR_15Oct2026.csv`, `TADPOLE`, and `PREPARE` across the ADNI study phases (`ADNIGO/ADNI2`, `ADNI3`, `ADNI4`).

In [2]:
# 1. Check DXSUM columns by PHASE
print("=== DXSUM_15Oct2026.csv: Non-Null Counts of Diagnosis Columns by ADNI Phase ===")
dx_cols = ["DIAGNOSIS", "DXNORM", "DXMCI", "DXAD", "DXMDUE", "DXDDUE", "DXDSEV"]
dxsum_phase_cov = dxsum_df.groupby("PHASE")[dx_cols].apply(lambda g: g.notna().sum() - (g == -4).sum())
dxsum_phase_cov["Total_Rows"] = dxsum_df.groupby("PHASE").size()
print(dxsum_phase_cov[["Total_Rows"] + dx_cols].to_string())

# 2. Check Master Group static nature
n_unique_grp_per_subj = master_df.groupby("Subject")["Group"].nunique()
print(f"\n=== Master 'Group' Column (from LONI IDA Search) ===")
print(f"Total unique subjects in Master: {master_df['Subject'].nunique():,}")
print(f"Subjects whose 'Group' changes across longitudinal visits: {(n_unique_grp_per_subj > 1).sum()} (0.00%)")
print("Master 'Group' value counts:")
print(master_df["Group"].value_counts(dropna=False).to_string())

# 3. Check ADNIMERGE phase coverage vs Master MRIProtocolPhase
print("\n=== Master MRIProtocolPhase vs ADNIMERGE COLPROT Counts ===")
print("Master MRIProtocolPhase:")
print(master_df["MRIProtocolPhase"].value_counts(dropna=False).to_string())
print("\nADNIMERGE COLPROT (and non-null DX count):")
am_cov = adnimerge_df.groupby("COLPROT").agg(Total_Rows=("DX", "size"), NonNull_DX=("DX", lambda s: s.notna().sum()))
print(am_cov.to_string())

=== DXSUM_15Oct2026.csv: Non-Null Counts of Diagnosis Columns by ADNI Phase ===
        Total_Rows  DIAGNOSIS  DXNORM  DXMCI  DXAD  DXMDUE  DXDDUE  DXDSEV
PHASE                                                                     
ADNI1         3868       3868    1130   1601  1129    1601       0    1129
ADNI2         5671       5671       0      0     0    2608    1107    1107
ADNI3         3587       3547       0      0     0    1258     440     440
ADNI4         2871       2865       0      0     0     856     308     306
ADNIGO         475        475       0      0     0     323      63      63
TEAM            74         74       0      0     0       0       0       0

=== Master 'Group' Column (from LONI IDA Search) ===
Total unique subjects in Master: 1,329
Subjects whose 'Group' changes across longitudinal visits: 0 (0.00%)
Master 'Group' value counts:
Group
CN           1433
MCI           888
EMCI          383
AD            244
LMCI          208
SMC           120
TEAM-ADNI      

## 2. Matching Algorithm: Assigning Visit-Specific `Diagnosis` from `DXSUM_15Oct2026.csv`

For each of the **3,278** scan sessions in `clean_dedup_fmri_to_t1_master.csv`, we resolve the clinical diagnosis at the time of the scan using the following hierarchy:
1. **Exact `(Subject == PTID, VISCODE2 == VISCODE2)` match in `DXSUM_15Oct2026.csv` (`2,918` rows):**
   - Matches the participant ID and longitudinal visit code (`bl`, `m06`, `m12`, `m24`, ..., `m180`). If duplicate rows exist for the same `(PTID, VISCODE2)`, ties are broken by exact `Visit == VISCODE` and minimum `|EXAMDATE - ScanDate|`.
2. **Exact `(Subject == PTID, Visit == VISCODE)` match in `DXSUM_15Oct2026.csv` (`4` rows):**
   - Handles rows where `VISCODE2` was `NaN` in `MRIQC` (`019_S_4293` at `Visit=y1`, `031_S_4005` at `Visit=v31`, and the 2 `TEAM-ADNI` rows at `Visit=t_bl`).
3. **Nearest-Date `DXSUM_15Oct2026.csv` match for MRI-only visits (`354` rows):**
   - In `ADNIGO/ADNI2`, `scmri` (`191` rows, Screening MRI acquired ~1–2 weeks between clinical `sc` and `bl`) and `m03` (`149` rows, Month-3 MRI-only visit where no clinical `DXSUM` form is administered) do not have a separate `scmri` or `m03` entry in `DXSUM`. Similarly, `14` `ADNI3/ADNI4` scans had unrecorded `DXSUM` forms at that specific visit code. Matching to the subject's nearest-in-time `DXSUM` evaluation (median offset `29` days) links `scmri` to `sc`/`bl` and `m03` to `bl`/`m06`.
4. **Fallback to `CDR_15Oct2026.csv` (`CDGLOBAL`) for 2 `TEAM-ADNI` rows (`DIAGNOSIS=10.0`) and 2 `ADNIGO` `scmri` screen-withdrawal subjects (`031_S_2017`, `130_S_2402`):**
   - Both `TEAM-ADNI` participants (`010_S_12018`, `070_S_12043`) have `CDGLOBAL = 0.0` $\rightarrow$ `CN`.
   - Both `ADNIGO` `scmri` participants (`031_S_2017`, `130_S_2402`) withdrew before `bl` (so no `DXSUM` record was created), and both have Screening `CDGLOBAL = 0.5` (`CDMEMORY = 0.5`) and LONI `Group = EMCI` $\rightarrow$ `MCI`.

In [3]:
# Parse scan date in master: StudyDate if present, else Acq Date
scan_date_dt = pd.to_datetime(master_df["StudyDate"], errors="coerce").fillna(
    pd.to_datetime(master_df["Acq Date"], errors="coerce")
)

# Prepare DXSUM valid rows
DX_CODE_MAP = {1.0: "CN", 2.0: "MCI", 3.0: "AD", 10.0: "TEAM_NODX"}
dxsum_valid = dxsum_df[dxsum_df["DIAGNOSIS"].notna()].copy()
dxsum_valid["DX_Label"] = dxsum_valid["DIAGNOSIS"].map(DX_CODE_MAP)
dxsum_valid["EXAMDATE_dt"] = pd.to_datetime(dxsum_valid["EXAMDATE"], errors="coerce").fillna(
    pd.to_datetime(dxsum_valid["USERDATE"], errors="coerce")
)
dx_by_ptid = {ptid: grp.copy() for ptid, grp in dxsum_valid.groupby("PTID")}

# Prepare CDR valid rows
cdr_valid = cdr_df[cdr_df["CDGLOBAL"].notna() & (cdr_df["CDGLOBAL"] >= 0)].copy()
cdr_valid["VISDATE_dt"] = pd.to_datetime(cdr_valid["VISDATE"], errors="coerce").fillna(
    pd.to_datetime(cdr_valid["USERDATE"], errors="coerce")
)
cdr_valid["CDR_DX"] = cdr_valid["CDGLOBAL"].apply(lambda v: "CN" if v == 0.0 else ("MCI" if v == 0.5 else "AD"))
cdr_by_ptid = {ptid: grp.copy() for ptid, grp in cdr_valid.groupby("PTID")}

# Prepare ADNIMERGE valid rows
ADNIMERGE_MAP = {"CN": "CN", "MCI": "MCI", "Dementia": "AD"}
adnimerge_df["DX_Clean"] = adnimerge_df["DX"].map(ADNIMERGE_MAP)
adnimerge_df["EXAMDATE_dt"] = pd.to_datetime(adnimerge_df["EXAMDATE"], errors="coerce")
am_by_ptid = {ptid: grp.copy() for ptid, grp in adnimerge_df.groupby("PTID")}

# Prepare TADPOLE and PREPARE lookups
tadpole_df["DX_Clean"] = tadpole_df["DX"].map(ADNIMERGE_MAP)
tad_by_ptid = {ptid: grp.copy() for ptid, grp in tadpole_df.groupby("PTID")}
prep_by_ptid = dict(zip(prepare_df["PTID"].astype(str), prepare_df["DX_bl"])) if "PTID" in prepare_df.columns and "DX_bl" in prepare_df.columns else {}

GROUP_TO_3CLASS = {
    "CN": "CN", "SMC": "CN",
    "EMCI": "MCI", "MCI": "MCI", "LMCI": "MCI",
    "AD": "AD", "TEAM-ADNI": "TEAM_NODX",
}

diag_records = []
for idx, row in master_df.iterrows():
    subj = row["Subject"]
    visit = str(row["Visit"]) if pd.notna(row["Visit"]) else ""
    vis2 = str(row["VISCODE2"]) if pd.notna(row["VISCODE2"]) else ""
    sdate = scan_date_dt.iloc[idx]
    grp_raw = row["Group"]
    grp_3class = GROUP_TO_3CLASS.get(grp_raw, grp_raw)

    # 1. DXSUM lookup
    dx_val = None
    dx_method = None
    dx_days = None
    sub_dx = dx_by_ptid.get(subj)

    if sub_dx is not None and len(sub_dx) > 0:
        m_vis2 = sub_dx[sub_dx["VISCODE2"].astype(str) == vis2] if vis2 else pd.DataFrame()
        m_vis1 = sub_dx[sub_dx["VISCODE"].astype(str) == visit] if visit else pd.DataFrame()

        if len(m_vis2) > 0:
            m_vis2 = m_vis2.assign(
                vis_exact=(m_vis2["VISCODE"].astype(str) == visit).astype(int),
                days=(m_vis2["EXAMDATE_dt"] - sdate).abs().dt.days.fillna(99999),
            ).sort_values(["vis_exact", "days", "ID"], ascending=[False, True, False])
            best = m_vis2.iloc[0]
            dx_val = best["DX_Label"]
            dx_method = "1_DXSUM_exact_VISCODE2"
            dx_days = best["days"] if best["days"] < 99999 else None
        elif len(m_vis1) > 0:
            m_vis1 = m_vis1.assign(
                days=(m_vis1["EXAMDATE_dt"] - sdate).abs().dt.days.fillna(99999),
            ).sort_values(["days", "ID"], ascending=[True, False])
            best = m_vis1.iloc[0]
            dx_val = best["DX_Label"]
            dx_method = "2_DXSUM_exact_VISCODE"
            dx_days = best["days"] if best["days"] < 99999 else None
        else:
            sub_dx_d = sub_dx.assign(
                days=(sub_dx["EXAMDATE_dt"] - sdate).abs().dt.days.fillna(99999),
            ).sort_values(["days", "ID"], ascending=[True, False])
            best = sub_dx_d.iloc[0]
            dx_val = best["DX_Label"]
            dx_method = "3_DXSUM_nearest_date"
            dx_days = best["days"] if best["days"] < 99999 else None

    # 2. CDR lookup (for comparison & fallback)
    cdr_dx = None
    cdr_score = None
    sub_cdr = cdr_by_ptid.get(subj)
    if sub_cdr is not None and len(sub_cdr) > 0:
        mc_vis2 = sub_cdr[sub_cdr["VISCODE2"].astype(str) == vis2] if vis2 else pd.DataFrame()
        mc_vis1 = sub_cdr[sub_cdr["VISCODE"].astype(str) == visit] if visit else pd.DataFrame()
        if len(mc_vis2) > 0:
            best_c = mc_vis2.iloc[-1]
        elif len(mc_vis1) > 0:
            best_c = mc_vis1.iloc[-1]
        else:
            sub_cdr_d = sub_cdr.assign(
                days=(sub_cdr["VISDATE_dt"] - sdate).abs().dt.days.fillna(99999)
            ).sort_values("days")
            best_c = sub_cdr_d.iloc[0]
        cdr_dx = best_c["CDR_DX"]
        cdr_score = best_c["CDGLOBAL"]

    # Resolve final diagnosis
    final_dx = dx_val
    final_method = dx_method
    if final_dx is None:
        final_dx = cdr_dx if cdr_dx is not None else grp_3class
        final_method = "4_CDR_and_Group_fallback_screen_withdrawal"
    elif final_dx == "TEAM_NODX":
        final_dx = cdr_dx if cdr_dx is not None else "CN"
        final_method = "4_TEAM_NODX_resolved_via_CDR"

    # 3. ADNIMERGE lookup (for comparison)
    am_dx_vis = None
    am_dx_nearest = None
    am_dx_bl = None
    sub_am = am_by_ptid.get(subj)
    if sub_am is not None and len(sub_am) > 0:
        am_dx_bl_raw = sub_am["DX_bl"].dropna().iloc[0] if len(sub_am["DX_bl"].dropna()) > 0 else None
        am_dx_bl = GROUP_TO_3CLASS.get(am_dx_bl_raw, am_dx_bl_raw)
        ma_vis = sub_am[(sub_am["VISCODE"].astype(str) == vis2) & sub_am["DX_Clean"].notna()]
        if len(ma_vis) > 0:
            am_dx_vis = ma_vis.iloc[0]["DX_Clean"]
        sub_am_valid = sub_am[sub_am["DX_Clean"].notna()].copy()
        if len(sub_am_valid) > 0:
            sub_am_valid["days"] = (sub_am_valid["EXAMDATE_dt"] - sdate).abs().dt.days.fillna(99999)
            best_am = sub_am_valid.sort_values("days").iloc[0]
            if best_am["days"] <= 180:
                am_dx_nearest = best_am["DX_Clean"]

    # 4. TADPOLE lookup (for comparison)
    tad_dx_vis = None
    sub_tad = tad_by_ptid.get(subj)
    if sub_tad is not None and len(sub_tad) > 0:
        mt_vis = sub_tad[(sub_tad["VISCODE"].astype(str) == vis2) & sub_tad["DX_Clean"].notna()]
        if len(mt_vis) > 0:
            tad_dx_vis = mt_vis.iloc[0]["DX_Clean"]

    diag_records.append({
        "Image Data ID": row["Image Data ID"],
        "Subject": subj,
        "Visit": visit,
        "VISCODE2": vis2,
        "StudyDate": row["StudyDate"],
        "MRIProtocolPhase": row["MRIProtocolPhase"],
        "LONI_Group_Raw": grp_raw,
        "LONI_Group_3Class": grp_3class,
        "Diagnosis": final_dx,
        "Match_Method": final_method,
        "DXSUM_Days_Apart": dx_days,
        "ADNIMERGE_DX_ExactVis": am_dx_vis,
        "ADNIMERGE_DX_Nearest180d": am_dx_nearest,
        "ADNIMERGE_DX_bl_3Class": am_dx_bl,
        "CDR_DX": cdr_dx,
        "CDR_Global": cdr_score,
        "TADPOLE_DX_ExactVis": tad_dx_vis,
    })

comp_df = pd.DataFrame(diag_records)

# Add Diagnosis column to master_df and save
master_df["Diagnosis"] = comp_df["Diagnosis"]
master_df.to_csv(MASTER_CSV, index=False)

print(f"Saved updated {MASTER_CSV.relative_to(PROJECT_ROOT)} with 'Diagnosis' column ({len(master_df):,} rows, {master_df['Diagnosis'].isna().sum()} NaNs).")
print("\n=== Final 'Diagnosis' Distribution (3,278 Scans) ===")
for k, v in master_df["Diagnosis"].value_counts().items():
    print(f"  {k:4s}: {v:4,d} ({100*v/len(master_df):.2f}%)")

print("\n=== Resolution Method Breakdown ===")
print(comp_df["Match_Method"].value_counts().to_string())

Saved updated results\dataset_creation\clean_dedup_fmri_to_t1_master.csv with 'Diagnosis' column (3,278 rows, 0 NaNs).

=== Final 'Diagnosis' Distribution (3,278 Scans) ===
  CN  : 1,538 (46.92%)
  MCI : 1,318 (40.21%)
  AD  :  422 (12.87%)

=== Resolution Method Breakdown ===
Match_Method
1_DXSUM_exact_VISCODE2                        2918
3_DXSUM_nearest_date                           354
4_TEAM_NODX_resolved_via_CDR                     2
2_DXSUM_exact_VISCODE                            2
4_CDR_and_Group_fallback_screen_withdrawal       2


## 3. Cross-Source Agreement & Conflict Analysis

Here we systematically compare our primary visit-level `Diagnosis` (`DXSUM_15Oct2026.csv`) against every other candidate source in the repository to quantify coverage, agreement, and the exact reasons for any conflicts.

In [4]:
# Summary table of all candidate sources: Coverage & Agreement with Final Diagnosis
sources_summary = []

def eval_source(name, series, why_not_used):
    non_null = series.notna() & (series != "TEAM_NODX")
    n_cov = int(non_null.sum())
    pct_cov = 100.0 * n_cov / len(comp_df)
    sub = comp_df[non_null]
    n_agree = int((sub["Diagnosis"] == series[non_null]).sum())
    n_conflict = n_cov - n_agree
    pct_agree = (100.0 * n_agree / n_cov) if n_cov > 0 else 0.0
    sources_summary.append({
        "Source / Column": name,
        "Covered_Scans": f"{n_cov:,} / {len(comp_df):,} ({pct_cov:.2f}%)",
        "Missing_Scans": f"{len(comp_df) - n_cov:,} ({100 - pct_cov:.2f}%)",
        "Agreement_When_Present": f"{n_agree:,} / {n_cov:,} ({pct_agree:.2f}%)",
        "Conflicts": n_conflict,
        "Role / Why Not Primary": why_not_used,
    })

eval_source(
    "1. DXSUM_15Oct2026.csv [DIAGNOSIS] (Selected Primary)",
    comp_df["Diagnosis"],
    "PRIMARY SOURCE: Official visit-level clinical diagnosis across ADNI1/GO/2/3/4 (2005-2026)"
)
eval_source(
    "2. ADNIMERGE_15Oct2026.csv [DX (exact visit)]",
    comp_df["ADNIMERGE_DX_ExactVis"],
    "Derived from DXSUM (100% agreement), but ends Sept 2023 (0 ADNI4 rows) & missing scmri/m03"
)
eval_source(
    "3. ADNIMERGE_15Oct2026.csv [DX (nearest <=180d)]",
    comp_df["ADNIMERGE_DX_Nearest180d"],
    "Derived from DXSUM, but missing 922 ADNI3/ADNI4 scans after Sept 2023"
)
eval_source(
    "4. Master / LONI IDA Search [Group (3-class)]",
    comp_df["LONI_Group_3Class"],
    "Static baseline enrollment group; misses 452 longitudinal conversions/reversions (13.8%)"
)
eval_source(
    "5. ADNIMERGE_15Oct2026.csv [DX_bl (3-class)]",
    comp_df["ADNIMERGE_DX_bl_3Class"],
    "Static baseline diagnosis + missing all new ADNI4 enrollees"
)
eval_source(
    "6. CDR_15Oct2026.csv [CDGLOBAL -> 0:CN, 0.5:MCI, >=1:AD]",
    comp_df["CDR_DX"],
    "Single staging scale, not consensus dx (135 AD have CDR=0.5; 125 MCI have CDR=0)"
)
eval_source(
    "7. tadpole_challenge/ADNIMERGE.csv [DX (exact visit)]",
    comp_df["TADPOLE_DX_ExactVis"],
    "Frozen 2017 challenge snapshot; missing 78.7% of scans (all 2017-2026 ADNI3/ADNI4)"
)
eval_source(
    "8. DXSUM_15Oct2026.csv [DXNORM / DXMCI / DXAD]",
    pd.Series([None] * len(comp_df)),
    "Legacy ADNI1-only columns; 0% populated in ADNIGO/ADNI2/ADNI3/ADNI4"
)

summary_table_df = pd.DataFrame(sources_summary)
print("=== Comparison of All Candidate Diagnosis Sources Across 3,278 Scans ===")
print(summary_table_df.to_string(index=False))

print("\n" + "="*95)
print("DETAIL A: Baseline LONI 'Group' vs. Visit-Specific 'Diagnosis' (452 Longitudinal Transitions)")
print("="*95)
ct_grp = pd.crosstab(comp_df["LONI_Group_Raw"], comp_df["Diagnosis"], margins=True)
print(ct_grp.to_string())

transitions = comp_df[
    (comp_df["LONI_Group_3Class"] != "TEAM_NODX") &
    (comp_df["LONI_Group_3Class"] != comp_df["Diagnosis"])
]
print(f"\nBreakdown of the {len(transitions)} scans where Visit 'Diagnosis' differs from Baseline 'Group':")
trans_counts = transitions.groupby(["LONI_Group_Raw", "LONI_Group_3Class", "Diagnosis"]).size().reset_index(name="Scan_Count")
for _, r in trans_counts.sort_values("Scan_Count", ascending=False).iterrows():
    direction = "Progression (Conversion)" if (
        (r["LONI_Group_3Class"] == "CN" and r["Diagnosis"] in ("MCI", "AD")) or
        (r["LONI_Group_3Class"] == "MCI" and r["Diagnosis"] == "AD")
    ) else "Reversion"
    print(f"  Baseline {r['LONI_Group_Raw']:4s} ({r['LONI_Group_3Class']:3s}) -> Visit {r['Diagnosis']:3s}: {r['Scan_Count']:3d} scans  [{direction}]")

print("\n" + "="*95)
print("DETAIL B: CDR Global Score (CDGLOBAL) vs. Consensus Clinical Diagnosis (DXSUM)")
print("="*95)
ct_cdr = pd.crosstab(comp_df["CDR_Global"], comp_df["Diagnosis"], margins=True)
print(ct_cdr.to_string())

=== Comparison of All Candidate Diagnosis Sources Across 3,278 Scans ===
                                         Source / Column           Covered_Scans   Missing_Scans  Agreement_When_Present  Conflicts                                                                     Role / Why Not Primary
   1. DXSUM_15Oct2026.csv [DIAGNOSIS] (Selected Primary) 3,278 / 3,278 (100.00%)       0 (0.00%) 3,278 / 3,278 (100.00%)          0  PRIMARY SOURCE: Official visit-level clinical diagnosis across ADNI1/GO/2/3/4 (2005-2026)
           2. ADNIMERGE_15Oct2026.csv [DX (exact visit)]  1,897 / 3,278 (57.87%)  1,381 (42.13%) 1,897 / 1,897 (100.00%)          0 Derived from DXSUM (100% agreement), but ends Sept 2023 (0 ADNI4 rows) & missing scmri/m03
        3. ADNIMERGE_15Oct2026.csv [DX (nearest <=180d)]  2,804 / 3,278 (85.54%)    474 (14.46%)  2,796 / 2,804 (99.71%)          8                      Derived from DXSUM, but missing 922 ADNI3/ADNI4 scans after Sept 2023
           4. Master / LONI IDA Sea